# External baselines for the camera-ready

This notebook trains the new baselines the reviewers asked for, on the **same split, labels and training recipe** as the
pipeline (`cluster_pipeline/scripts/04_ff_best_combo.py`). It also retrains the two existing models it compares against, so every row in
the final table comes from one run.

| Model | Role |
|---|---|
| Stitched panorama (Model 1, pipeline defaults) | existing pixel-level baseline, retrained |
| Feature Fusion (best combo: d=5, w=1.5×, GELU, head 512, dropout 0.1) | the paper's model, retrained for a same-run comparison |
| Output Fusion (Baseline 2 defaults) | existing decision-level baseline, retrained |
| **Single-camera** | each direction predicted from its own tile only. Same backbone/head size as FF, no cross-camera input |
| **Permutation-equivariant fusion** | shared per-tile encoder + mean/max pooling across cameras. Shuffle-robust by construction |
| **Threshold (non-learning)** | hand-crafted temperature rules, thresholds grid-searched on train |

**`MODELS_TO_TRAIN`** (first code cell) picks which models run. To add only the stitched baseline to an earlier run,
set it to `["Stitched"]` and attach that run's output so `label_cache.pkl` is reused.

Every learned model is trained with 3 seeds and reported as mean ± std. The notebook also re-runs the shuffle test (all 5 non-identity
camera permutations) for every model, and saves per-frame val predictions for the failure-case analysis.

**Attach:** `janahussien90/rgb3thermals-full`, `janahussien90/homography-mats`. Also attach a previous run's output
(`label_cache.pkl`), if you have one, to skip YOLO. **Accelerator:** GPU T4. **Runtime:** about 1–1.5 h (YOLO about 25 min, tiles about 10 min,
training about 30–40 min). Use *Save Version → Save & Run All* so it runs in the background.

**Rerunning only the threshold baselines** (the current setting of `MODELS_TO_TRAIN`): attach `rgb3thermals-full`,
`homography-mats` and `label-cache`. No GPU is needed and it takes about 20 minutes. The output
`baselines_table.csv` then holds only the two threshold rows; the learned models' rows from the earlier run are unchanged.

Set `SMOKE_TEST = True` in the first cell first. It runs every cell on ~650 rows with 1 epoch and 1 seed in a few minutes, to catch errors before the long run.

In [ ]:
!pip install -q ultralytics

import copy, itertools, json, pickle, random, re, shutil, time, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.path import Path as MplPath
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset

# ---- identical to the pipeline ----
SEED = 42
VAL_FRAC = 0.2
N_SPLIT_BLOCKS = 10
MAX_RGB_GAP_MS = 150
YOLO_WEIGHTS = "yolov8x.pt"
COCO_PERSON_CLASS_ID = 0
SENSOR_W, SENSOR_H = 32, 24
ACTIVE_CAMERAS = [2, 5, 4]                  # left, middle, right
DIRECTIONS = ["left", "middle", "right"]
PANORAMA_REFERENCE_CAM = 5
FLIP_THERMAL_HORIZONTALLY = True
NUM_CAMS, NUM_CLASSES = 3, 3
CLASS_NAMES = ["clear", "person", "other"]
BATCH_SIZE, EPOCHS, LR = 64, 30, 1e-3
CLASS_WEIGHT_CAP = 20.0
BEST_FF = dict(depth=5, width_mult=1.5, act="gelu", head_hidden=512, head_dropout=0.1)

# ---- this notebook ----
# Which models to run. To only add one model to an earlier run, e.g.: MODELS_TO_TRAIN = ["Output Fusion (FF config)"]
# Full run: ["Stitched", "Feature Fusion", "Output Fusion", "Output Fusion (FF config)", "Single-camera", "Perm.-equivariant",
#            "Threshold (non-learning)", "Threshold (per-direction)"]
MODELS_TO_TRAIN = ["Threshold (non-learning)", "Threshold (per-direction)"]
TRAIN_SEEDS = [42, 43, 44]
YOLO_BATCH = 32
# ---- smoke test: ~600 rows, 1 epoch, 1 seed -- checks every cell end-to-end in a few minutes ----
SMOKE_TEST = False
SMOKE_N_TRAIN, SMOKE_N_VAL = 500, 150
if SMOKE_TEST:
    EPOCHS, TRAIN_SEEDS = 1, [42]
    print("*** SMOKE TEST: numbers are meaningless -- set SMOKE_TEST = False for the real run ***")

OUT = Path("/kaggle/working")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Data, split, geometry

This is the same discovery and split as `build_sample.ipynb`, which are line-for-line copies of the pipeline.

In [ ]:
import os, zipfile

DATASET = "full"            # "sample" -> rgb3thermals_sample/pictureData, "full" -> rgb3thermals_full/all_runs
PICTURE_DIR_NAMES = {"sample": "pictureData", "full": "all_runs"}
CALIBRATION_NAMES = ["thermal_homography_mats", "rgb homography final"]
KAGGLE_INPUT = Path("/kaggle/input")
EXTRACT_TO = Path("/kaggle/working/rgb3thermals_extracted")   # /kaggle/input is read-only
MAX_SEARCH_DEPTH = 6


def _is_capture_dir(name):
    return bool(re.fullmatch(r"part\d+", name)) or name.startswith(("thermal-g", "rgb-g", "sync-g"))


def _walk_dirs(root, max_depth=MAX_SEARCH_DEPTH):
    stack = [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = list(os.scandir(d))
        except (PermissionError, NotADirectoryError):
            continue
        yield d, {e.name for e in entries}
        if depth < max_depth:
            for e in entries:
                if e.is_dir() and e.name != "__MACOSX" and not _is_capture_dir(e.name):
                    stack.append((Path(e.path), depth + 1))


def find_base(names, search_root=KAGGLE_INPUT, hint=""):
    """The single folder under search_root that contains every name in `names` (as a folder or .zip)."""
    complete = sorted(d for d, entries in _walk_dirs(search_root)
                      if all(n in entries or f"{n}.zip" in entries for n in names))
    if not complete:
        raise FileNotFoundError(f"No folder under {search_root} contains all of {names} (folder or .zip). {hint}")
    base = min(complete, key=lambda d: len(d.parts))
    others = [d for d in complete if d != base and base not in d.parents]
    if others:
        raise ValueError(f"Multiple candidate folders for {names}: {[str(base)] + [str(d) for d in others]}")
    return base


def ensure_extracted(base, name):
    """base/<name> if it is a folder, else extracts base/<name>.zip into EXTRACT_TO/<name> (skipping macOS junk)."""
    for candidate in [base / name, EXTRACT_TO / name]:
        if candidate.is_dir():
            print(f"using folder: {candidate}")
            return candidate
    zip_path = base / f"{name}.zip"
    if not zip_path.is_file():
        raise FileNotFoundError(f"Neither folder '{name}' nor '{name}.zip' found in {base}")
    EXTRACT_TO.mkdir(parents=True, exist_ok=True)
    dest, tmp = EXTRACT_TO / name, EXTRACT_TO / f".{name}.extracting"
    if tmp.exists():
        shutil.rmtree(tmp)
    print(f"extracting {zip_path.name} -> {dest} ...")
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if not m.startswith("__MACOSX/") and not Path(m).name.startswith("._")]
        zf.extractall(tmp, members=members)
    tmp.rename(dest)
    return dest


CALIB_BASE = find_base(CALIBRATION_NAMES, hint="Is janahussien90/homography-mats attached?")
PICTURE_BASE = find_base([PICTURE_DIR_NAMES[DATASET]], hint="Is janahussien90/rgb3thermals-full attached?")
DATA_ROOT = ensure_extracted(PICTURE_BASE, PICTURE_DIR_NAMES[DATASET])
HOMOGRAPHY_DIR = ensure_extracted(CALIB_BASE, "thermal_homography_mats")
RGB_HOMOGRAPHY_DIR = ensure_extracted(CALIB_BASE, "rgb homography final")


def _part_sort_key(p):
    m = re.search(r"(\d+)$", p.name)
    return int(m.group(1)) if m else 0


def _find_one(root, prefix):
    matches = sorted(p for p in Path(root).glob(f"{prefix}-g*") if p.is_dir())
    if len(matches) > 1:
        warnings.warn(f"{Path(root).name}: several '{prefix}-g*' folders, using {matches[0].name}")
    return matches[0] if matches else None


def _find_part_dirs(root, max_depth=3):
    """partN folders may sit directly in DATA_ROOT or a level or two deeper (zip wrapper folders)."""
    found, stack = [], [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        for e in os.scandir(d):
            if not e.is_dir() or e.name == "__MACOSX":
                continue
            if re.fullmatch(r"part\d+", e.name):
                found.append(Path(e.path))
            elif depth < max_depth:
                stack.append((Path(e.path), depth + 1))
    return found


# Same order as the pipeline (numeric part order) -- the split depends on it.
CAPTURE_PARTS, SKIPPED_PARTS = {}, {}
for root in sorted(_find_part_dirs(DATA_ROOT), key=_part_sort_key):
    paths = {"thermal_dir": _find_one(root, "thermal"), "rgb_dir": _find_one(root, "rgb"), "sync_dir": _find_one(root, "sync")}
    missing = [k for k, v in paths.items() if v is None]
    if not missing and not (paths["sync_dir"] / "sync.csv").is_file():
        missing = ["sync.csv"]
    if missing:
        SKIPPED_PARTS[root.name] = missing
        continue
    paths["sync_csv"] = paths.pop("sync_dir") / "sync.csv"
    CAPTURE_PARTS[root.name] = paths
if SKIPPED_PARTS:
    warnings.warn(f"Skipping incomplete parts (the cluster pipeline would have crashed on these, "
                  f"so the split may differ from the paper's): {SKIPPED_PARTS}")
print("captures:   ", DATA_ROOT)
print("thermal H:  ", HOMOGRAPHY_DIR)
print("rgb H:      ", RGB_HOMOGRAPHY_DIR)
print(f"{len(CAPTURE_PARTS)} parts: {list(CAPTURE_PARTS)}")


all_rows = []
for part, paths in CAPTURE_PARTS.items():
    rows = pd.read_csv(paths["sync_csv"]).to_dict("records")
    for i, row in enumerate(rows):
        row["_part"] = part
        row.setdefault("anchor_index", i)
    all_rows.extend(rows)
random.Random(SEED).shuffle(all_rows)

def blocked_train_val_split(rows, val_frac=VAL_FRAC, n_blocks=N_SPLIT_BLOCKS):
    n = len(rows)
    block_size = max(1, n // n_blocks)
    train_rows, val_rows = [], []
    for start in range(0, n, block_size):
        block = rows[start:start + block_size]
        n_val = max(1, int(len(block) * val_frac)) if len(block) > 1 else 0
        if n_val == 0:
            train_rows.extend(block)
        else:
            train_rows.extend(block[:-n_val])
            val_rows.extend(block[-n_val:])
    if not val_rows:
        val_rows = train_rows[-1:]
    return train_rows, val_rows

def rgb_gap_ok(row, max_gap_ms=MAX_RGB_GAP_MS):
    files = row.get("rgb_files")
    if not isinstance(files, str) or not files:
        return False
    try:
        return 0 <= float(row.get("rgb_gap_ms", -1)) <= max_gap_ms
    except (ValueError, TypeError):
        return False

train_rows, val_rows = blocked_train_val_split(all_rows)
if SMOKE_TEST:
    train_rows, val_rows = train_rows[:SMOKE_N_TRAIN], val_rows[:SMOKE_N_VAL]
print(f"total sync rows: {len(all_rows)}   train: {len(train_rows)}   val: {len(val_rows)}")

CAM_NAME_RE = re.compile(r"cam(\d+)")

def _corners(w, h, T):
    c = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], float)
    out = (T @ np.hstack([c, np.ones((4, 1))]).T).T
    return out[:, :2] / out[:, 2:3]

homographies = []
for path in sorted(HOMOGRAPHY_DIR.rglob("*homography*.json")):
    if path.name.startswith("._") or "__MACOSX" in path.parts:
        continue
    d = json.loads(path.read_text())
    a, b = CAM_NAME_RE.match(d["cam_a"]), CAM_NAME_RE.match(d["cam_b"])
    homographies.append({"cam_a": int(a.group(1)), "cam_b": int(b.group(1)), "H": np.array(d["H"], float)})
graph = {c: [] for c in ACTIVE_CAMERAS}
for e in homographies:
    if e["cam_a"] in graph and e["cam_b"] in graph:
        graph[e["cam_a"]].append((e["cam_b"], np.linalg.inv(e["H"])))
        graph[e["cam_b"]].append((e["cam_a"], e["H"]))
CAM_TRANSFORMS, queue = {PANORAMA_REFERENCE_CAM: np.eye(3)}, [PANORAMA_REFERENCE_CAM]
while queue:
    cur = queue.pop(0)
    for nb, T_nb in graph[cur]:
        if nb not in CAM_TRANSFORMS:
            CAM_TRANSFORMS[nb] = CAM_TRANSFORMS[cur] @ T_nb
            queue.append(nb)
pts = np.vstack([_corners(SENSOR_W, SENSOR_H, T) for T in CAM_TRANSFORMS.values()])
min_x, min_y = np.floor(pts.min(0)).astype(int)
TRANSLATION = np.array([[1, 0, -min_x], [0, 1, -min_y], [0, 0, 1]], float)
CAM_FOOTPRINTS = {c: _corners(SENSOR_W, SENSOR_H, TRANSLATION @ T) for c, T in CAM_TRANSFORMS.items()}

RGB_CALIBRATIONS = {}
for path in sorted(RGB_HOMOGRAPHY_DIR.rglob("rgb*_homography.json")):
    if path.name.startswith("._") or "__MACOSX" in path.parts:
        continue
    d = json.loads(path.read_text())
    cam = int(CAM_NAME_RE.match(d["cam_b"]).group(1))
    RGB_CALIBRATIONS[cam] = {"H": np.array(d["H"], float), "centroid": np.array(d["quad_a"], float).mean(0)}
assert set(ACTIVE_CAMERAS) <= set(RGB_CALIBRATIONS) & set(CAM_TRANSFORMS)

## 2. Labels for every row

This is the same rule as the pipeline's `generate_real_labels`. YOLOv8x runs in batches, and the result is cached to `/kaggle/working/label_cache.pkl`.
If a `label_cache.pkl` from an earlier run is attached as input, it is loaded and YOLO is skipped.

In [ ]:
def _find_file(name, root=Path("/kaggle/input")):
    hits = sorted(root.rglob(name)) if root.exists() else []
    return hits[0] if hits else None

LABEL_CACHE = {}          # (part, anchor_index) -> (class_vec[3], mask_vec[3])
_prev = Path("/kaggle/input/datasets/janahussien90/label-cache/label_cache.pkl")
_prev = _prev if _prev.exists() else _find_file("label_cache.pkl")
if _prev:
    LABEL_CACHE = pickle.load(open(_prev, "rb"))
    print(f"loaded {len(LABEL_CACHE)} cached labels from {_prev}")

FOOTPRINT_PATHS = {c: MplPath(CAM_FOOTPRINTS[c]) for c in ACTIVE_CAMERAS}

def labels_from_result(res):
    person = {c: False for c in ACTIVE_CAMERAS}
    other = dict(person)
    for box, cls in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = box
        center = np.array([(x1 + x2) / 2, (y1 + y2) / 2])
        via = min(RGB_CALIBRATIONS, key=lambda c: np.linalg.norm(center - RGB_CALIBRATIONS[c]["centroid"]))
        corners = np.array([[x1, y1, 1], [x2, y1, 1], [x2, y2, 1], [x1, y2, 1]], float)
        proj = (TRANSLATION @ CAM_TRANSFORMS[via] @ RGB_CALIBRATIONS[via]["H"] @ corners.T).T
        proj = proj[:, :2] / proj[:, 2:3]
        pts = list(map(tuple, proj)) + [tuple(proj.mean(0))]
        is_person = int(cls) == COCO_PERSON_CLASS_ID
        for c in ACTIVE_CAMERAS:
            if any(FOOTPRINT_PATHS[c].contains_point(p) for p in pts):
                (person if is_person else other)[c] = True
    return np.array([1 if person[c] else 2 if other[c] else 0 for c in ACTIVE_CAMERAS], dtype=np.int64)

key_of = lambda r: (r["_part"], int(r["anchor_index"]))
todo = [r for r in train_rows + val_rows if rgb_gap_ok(r) and key_of(r) not in LABEL_CACHE]
print(f"{len(todo)} rows need YOLO")
if todo:
    from ultralytics import YOLO
    yolo = YOLO(YOLO_WEIGHTS)
    read = lambda r: cv2.imread(str(CAPTURE_PARTS[r["_part"]]["rgb_dir"] / r["rgb_files"]))
    t0 = time.time()
    with ThreadPoolExecutor(8) as pool:
        for s in range(0, len(todo), YOLO_BATCH):
            batch = todo[s:s + YOLO_BATCH]
            imgs = list(pool.map(read, batch))
            ok = [(r, im) for r, im in zip(batch, imgs) if im is not None]
            for (r, _), res in zip(ok, yolo([im for _, im in ok], verbose=False)):
                LABEL_CACHE[key_of(r)] = (labels_from_result(res), np.ones(NUM_CAMS, np.float32))
            if (s // YOLO_BATCH) % 50 == 0:
                print(f"  {s + len(batch)}/{len(todo)}  ({time.time() - t0:.0f}s)")
    pickle.dump(LABEL_CACHE, open(OUT / "label_cache.pkl", "wb"))
    print(f"saved {len(LABEL_CACHE)} labels -> {OUT / 'label_cache.pkl'}")

def get_label(row):
    if rgb_gap_ok(row) and key_of(row) in LABEL_CACHE:
        return LABEL_CACHE[key_of(row)]
    return np.zeros(NUM_CAMS, np.int64), np.zeros(NUM_CAMS, np.float32)   # masked, as in the pipeline

## 3. Thermal tiles

This is the pipeline's `build_camera_sequence` (sticky fill-forward, horizontal flip). It loads every row's three tiles into memory.

In [ ]:
CAPTURE_RE = re.compile(r"^data_capture_(\d+)_(\d+)_(\d+)\.json$")

def _read_raw(path):
    text = path.read_text()
    if "nan" in text:                       # only the files that need the -nan fix pay for the regex
        text = re.sub(r"-nan\b", "NaN", text)
    return np.array(json.loads(text)["data"], dtype=np.float64)

def build_part_sequences(folder, cams, pool):
    """Same as the pipeline's build_camera_sequence (sticky fill-forward, horizontal flip), but lists the
    folder once for all cameras and reads the files in parallel. Output is identical."""
    caps = {c: [] for c in cams}
    for f in os.scandir(folder):
        m = CAPTURE_RE.match(f.name)
        if m and int(m.group(1)) in caps:
            caps[int(m.group(1))].append((int(m.group(2)), Path(f.path)))
    out = {}
    for c in cams:
        paths = [p for _, p in sorted(caps[c], key=lambda t: t[0])]
        seq, buffer = [], np.zeros(SENSOR_W * SENSOR_H)
        for raw in pool.map(_read_raw, paths):   # map keeps file order, so fill-forward is unchanged
            ok = (raw != 0) & ~np.isnan(raw)
            buffer[ok] = raw[ok]
            grid = buffer.copy().reshape(SENSOR_H, SENSOR_W)
            if FLIP_THERMAL_HORIZONTALLY:
                grid = np.ascontiguousarray(grid[:, ::-1])
            seq.append(grid.astype(np.float32))
        out[c] = seq
    return out

t0 = time.time()
SEQS = {}
needed_parts = {r["_part"] for r in train_rows + val_rows}
with ThreadPoolExecutor(16) as pool:
    for part, paths in CAPTURE_PARTS.items():
        if part not in needed_parts:
            continue
        SEQS[part] = build_part_sequences(paths["thermal_dir"], ACTIVE_CAMERAS, pool)
        print(f"  {part}: {[len(SEQS[part][c]) for c in ACTIVE_CAMERAS]} frames  ({time.time() - t0:.0f}s)")

def to_arrays(rows):
    X = np.zeros((len(rows), NUM_CAMS, 1, SENSOR_H, SENSOR_W), np.float32)
    Y = np.zeros((len(rows), NUM_CAMS), np.int64)
    M = np.zeros((len(rows), NUM_CAMS), np.float32)
    keep = np.ones(len(rows), bool)
    for i, r in enumerate(rows):
        part, a = key_of(r)
        try:
            X[i, :, 0] = np.stack([SEQS[part][c][a] for c in ACTIVE_CAMERAS])
        except IndexError:
            keep[i] = False
            continue
        Y[i], M[i] = get_label(r)
    if (~keep).any():
        warnings.warn(f"{(~keep).sum()} rows have an anchor past the end of a thermal sequence -- dropped")
    return X[keep], Y[keep], M[keep], [r for r, k in zip(rows, keep) if k]

Xtr, Ytr, Mtr, train_rows_k = to_arrays(train_rows)
Xva, Yva, Mva, val_rows_k = to_arrays(val_rows)
print(f"train {Xtr.shape}  val {Xva.shape}  labeled cells: train {int(Mtr.sum())}  val {int(Mva.sum())}")

# inverse-frequency class weight over train+val, capped -- exactly as the pipeline computes it
counts = np.array([((np.concatenate([Ytr, Yva]) == k) & (np.concatenate([Mtr, Mva]) > 0)).sum() for k in range(3)], float)
class_weight = torch.tensor(np.minimum(counts.sum() / (np.maximum(counts, 1) * NUM_CLASSES), CLASS_WEIGHT_CAP), dtype=torch.float32)
print("class counts:", dict(zip(CLASS_NAMES, counts.astype(int))), " weight:", class_weight.numpy().round(2))
json.dump({"parts": list(CAPTURE_PARTS), "n_sync_rows": len(all_rows), "n_train": len(train_rows_k),
           "n_val": len(val_rows_k), "class_counts": dict(zip(CLASS_NAMES, counts.astype(int).tolist()))},
          open(OUT / "split_info.json", "w"), indent=1)

## 4. Models

`FlexBackbone`, `FlexClassHead3Class`, `FeatureFusionModel`, `PerCameraHead` and `OutputFusionModel` are copied verbatim from the pipeline. The two new models come after them.

In [ ]:
ACT_MAP = {"relu": nn.ReLU, "gelu": nn.GELU, "leakyrelu": lambda: nn.LeakyReLU(0.1)}

def make_conv_block(in_ch, out_ch, act_fn):
    return nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), act_fn())

class FlexBackbone(nn.Module):
    BASE_CHANNELS = [16, 32, 64, 128, 256]

    def __init__(self, depth=3, width_mult=1.0, act="relu"):
        super().__init__()
        act_fn = ACT_MAP[act]
        channels = [max(4, int(c * width_mult)) for c in self.BASE_CHANNELS[:depth]]
        self.blocks = nn.ModuleList()
        in_ch = 1
        for i, out_ch in enumerate(channels):
            layers = [make_conv_block(in_ch, out_ch, act_fn)]
            if i < depth - 1:
                layers.append(nn.MaxPool2d(2))
            self.blocks.append(nn.Sequential(*layers))
            in_ch = out_ch
        self.out_channels = channels

    def forward(self, x):
        feats = []
        for block in self.blocks:
            x = block(x)
            feats.append(x)
        return feats

class FlexClassHead3Class(nn.Module):
    def __init__(self, feature_dims, hidden=128, dropout=0.3, n_out_per_cam=NUM_CLASSES):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.n_out_per_cam = n_out_per_cam
        self.fc = nn.Sequential(nn.Linear(sum(feature_dims), hidden), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(hidden, NUM_CAMS * n_out_per_cam))

    def forward(self, features):
        out = self.fc(torch.cat([self.pool(f).flatten(1) for f in features], dim=1))
        return out.view(-1, NUM_CAMS, self.n_out_per_cam)

class FeatureFusionModel(nn.Module):
    def __init__(self, depth=3, width_mult=1.0, act="relu", head_hidden=128, head_dropout=0.3):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        per_dims = self.backbone.out_channels
        self.proj = nn.ModuleList([nn.Conv2d(d * NUM_CAMS, d, 1) for d in per_dims])
        self.head = FlexClassHead3Class(per_dims, hidden=head_hidden, dropout=head_dropout)

    def forward(self, x):
        all_feats = [[] for _ in self.backbone.out_channels]
        for i in range(NUM_CAMS):
            for lvl, f in enumerate(self.backbone(x[:, i])):
                all_feats[lvl].append(f)
        return self.head([proj(torch.cat(lf, dim=1)) for proj, lf in zip(self.proj, all_feats)])

class PerCameraHead(nn.Module):
    def __init__(self, feature_dims, hidden=64, dropout=0.3, n_out=NUM_CLASSES):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(sum(feature_dims), hidden), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(hidden, n_out))

    def forward(self, features):
        return self.fc(torch.cat([self.pool(f).flatten(1) for f in features], dim=1))

class OutputFusionModel(nn.Module):
    def __init__(self, depth=3, width_mult=1.0, act="relu", head_hidden=64, head_dropout=0.3, fusion_hidden=64):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        per_dims = self.backbone.out_channels
        self.per_cam_heads = nn.ModuleList([PerCameraHead(per_dims, head_hidden, head_dropout) for _ in range(NUM_CAMS)])
        fusion_in = NUM_CAMS * NUM_CLASSES
        self.fusion_mlp = nn.Sequential(nn.Linear(fusion_in, fusion_hidden), nn.ReLU(), nn.Dropout(head_dropout),
                                        nn.Linear(fusion_hidden, fusion_in))

    def forward(self, x):
        logits = torch.cat([self.per_cam_heads[i](self.backbone(x[:, i])) for i in range(NUM_CAMS)], dim=1)
        return self.fusion_mlp(logits).view(-1, NUM_CAMS, NUM_CLASSES)

# ---------------- new baselines ----------------

class SingleCameraModel(nn.Module):
    """Each direction is predicted from its own tile only: shared backbone, one head per direction,
    no information crosses between cameras. Output Fusion without its fusion MLP, at FF's size."""

    def __init__(self, depth=3, width_mult=1.0, act="relu", head_hidden=128, head_dropout=0.3):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        per_dims = self.backbone.out_channels
        self.heads = nn.ModuleList([PerCameraHead(per_dims, head_hidden, head_dropout) for _ in range(NUM_CAMS)])

    def forward(self, x):
        return torch.stack([self.heads[i](self.backbone(x[:, i])) for i in range(NUM_CAMS)], dim=1)

class PermEquivariantModel(nn.Module):
    """DeepSets-style fusion: a shared encoder per tile, a context vector from mean and max pooling across
    cameras, and one shared head applied to [own features, context]. Permuting the input tiles permutes the
    outputs identically, so it is shuffle-robust by construction and needs no camera-identity classifier."""

    def __init__(self, depth=3, width_mult=1.0, act="relu", head_hidden=128, head_dropout=0.3):
        super().__init__()
        self.backbone = FlexBackbone(depth, width_mult, act)
        d = sum(self.backbone.out_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Linear(3 * d, head_hidden), nn.ReLU(), nn.Dropout(head_dropout),
                                  nn.Linear(head_hidden, NUM_CLASSES))

    def forward(self, x):
        B, C = x.shape[:2]
        z = torch.cat([self.pool(f).flatten(1) for f in self.backbone(x.flatten(0, 1))], 1).view(B, C, -1)
        ctx = torch.cat([z.mean(1), z.amax(1)], -1).unsqueeze(1).expand(-1, C, -1)
        return self.head(torch.cat([z, ctx], -1))

MODEL_SPECS = {
    "Feature Fusion":     lambda: FeatureFusionModel(**BEST_FF),
    "Output Fusion":      lambda: OutputFusionModel(),
    # capacity-matched decision-level fusion: Feature Fusion's exact backbone + head settings
    "Output Fusion (FF config)": lambda: OutputFusionModel(**BEST_FF),
    "Single-camera":      lambda: SingleCameraModel(**BEST_FF),
    "Perm.-equivariant":  lambda: PermEquivariantModel(**BEST_FF),
}

# sanity: shapes, and equivariance of the new fusion model
_x = torch.randn(4, NUM_CAMS, 1, SENSOR_H, SENSOR_W)
for name, mk in MODEL_SPECS.items():
    m = mk().eval()
    print(f"{name:20s} out {tuple(m(_x).shape)}  params {sum(p.numel() for p in m.parameters()):,}")
_m = MODEL_SPECS["Perm.-equivariant"]().eval()
with torch.no_grad():
    assert torch.allclose(_m(_x[:, [2, 0, 1]]), _m(_x)[:, [2, 0, 1]], atol=1e-5), "not equivariant"
print("perm.-equivariant model: equivariance check passed")

## 4b. Stitched panorama (Model 1)

This is ported verbatim from `cluster_pipeline/scripts/01_baseline1.py`. The pipeline stitches the panorama in **cam 4's** frame
(`CLUSTER_BUILD_ORDER = [4, 5, 2]`), so this model's geometry is rebuilt exactly that way. The labels above use cam 5's frame,
which doesn't affect them (point-in-footprint tests give the same answer in any common frame), but it does change the panorama this model sees.
Input: `[2, H, W]` = temperature + validity mask. One shared CNN runs on each camera's footprint bounding-box crop.

In [ ]:
CLUSTER_BUILD_ORDER = [4, 5, 2]

def _transform_corners(width, height, T):
    corners = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=np.float64)
    corners_h = np.hstack([corners, np.ones((4, 1))])
    transformed = (T @ corners_h.T).T
    return transformed[:, :2] / transformed[:, 2:3]

def compute_canvas_bounds(frame_shapes, transforms):
    all_points = np.vstack([_transform_corners(w, h, T) for (h, w), T in zip(frame_shapes, transforms)])
    min_x, min_y = np.floor(all_points.min(axis=0)).astype(int)
    max_x, max_y = np.ceil(all_points.max(axis=0)).astype(int)
    return min_x, min_y, max_x, max_y

def build_clusters(homography_entries, cam_ids):
    graph = {cam: [] for cam in cam_ids}
    for entry in homography_entries:
        a, b, H = entry["cam_a"], entry["cam_b"], entry["H"]
        if a not in graph or b not in graph:
            continue
        graph[a].append((b, np.linalg.inv(H)))
        graph[b].append((a, H))
    visited, clusters = set(), []
    for root_cam in cam_ids:
        if root_cam in visited:
            continue
        visited.add(root_cam)
        cams, transforms = [root_cam], [np.eye(3)]
        transform_to_root = {root_cam: np.eye(3)}
        queue = [root_cam]
        while queue:
            current = queue.pop(0)
            for neighbor, T_neighbor_to_current in graph[current]:
                if neighbor in visited:
                    continue
                visited.add(neighbor)
                transform_to_root[neighbor] = transform_to_root[current] @ T_neighbor_to_current
                cams.append(neighbor)
                transforms.append(transform_to_root[neighbor])
                queue.append(neighbor)
        clusters.append({"cams": cams, "transforms": transforms})
    return clusters

def warp_image(img, H, output_size):
    return cv2.warpPerspective(img, H, output_size, flags=cv2.INTER_NEAREST,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def stitch_frames(frames, transforms):
    shapes = [f.shape for f in frames]
    min_x, min_y, max_x, max_y = compute_canvas_bounds(shapes, transforms)
    canvas_w, canvas_h = max_x - min_x, max_y - min_y
    translation = np.array([[1, 0, -min_x], [0, 1, -min_y], [0, 0, 1]], dtype=np.float64)
    canvas = np.zeros((canvas_h, canvas_w), dtype=frames[0].dtype)
    for frame, T in zip(frames, transforms):
        warped = warp_image(frame, translation @ T, (canvas_w, canvas_h))
        mask = warped > 0
        canvas[mask] = warped[mask]
    return canvas

main_cluster = next((c for c in build_clusters(homographies, CLUSTER_BUILD_ORDER)
                     if len(c["cams"]) == NUM_CAMS and set(c["cams"]) == set(ACTIVE_CAMERAS)), None)
assert main_cluster is not None, "cams 2/5/4 are not one connected homography cluster"
PANORAMA_MIN_X, PANORAMA_MIN_Y, PANORAMA_MAX_X, PANORAMA_MAX_Y = compute_canvas_bounds(
    [(SENSOR_H, SENSOR_W)] * len(main_cluster["cams"]), main_cluster["transforms"])
PANORAMA_W, PANORAMA_H = PANORAMA_MAX_X - PANORAMA_MIN_X, PANORAMA_MAX_Y - PANORAMA_MIN_Y
PANORAMA_TRANSLATION = np.array([[1, 0, -PANORAMA_MIN_X], [0, 1, -PANORAMA_MIN_Y], [0, 0, 1]], dtype=np.float64)
STITCH_BBOX = {}
for cam_id, T in zip(main_cluster["cams"], main_cluster["transforms"]):
    corners = _transform_corners(SENSOR_W, SENSOR_H, PANORAMA_TRANSLATION @ T)
    STITCH_BBOX[cam_id] = (corners[:, 0].min(), corners[:, 1].min(), corners[:, 0].max(), corners[:, 1].max())
print("cluster cams (BFS order from cam 4):", main_cluster["cams"], f"  panorama {PANORAMA_H} x {PANORAMA_W}")

def pad_or_crop_to_canvas(panorama, target_h=None, target_w=None):
    target_h = target_h or PANORAMA_H
    target_w = target_w or PANORAMA_W
    h, w = panorama.shape
    out = np.zeros((target_h, target_w), dtype=panorama.dtype)
    copy_h, copy_w = min(h, target_h), min(w, target_w)
    out[:copy_h, :copy_w] = panorama[:copy_h, :copy_w]
    return out

def load_panorama(part, anchor_index):
    frames_all = [SEQS[part][c][anchor_index].astype(np.float64) for c in main_cluster["cams"]]
    panorama = stitch_frames(frames_all, [PANORAMA_TRANSLATION @ T for T in main_cluster["transforms"]])
    return pad_or_crop_to_canvas(panorama)

def build_stitched_input_tensor(panorama):
    mask = (panorama > 0).astype(np.float32)
    return np.stack([panorama.astype(np.float32), mask], axis=0)

class ThermalBackbone(nn.Module):
    def __init__(self, in_channels=2, pool_size=(3, 4)):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, 5, padding=2); self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1);          self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1);          self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1);          self.bn4 = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 64, 3, padding=1);          self.bn5 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.pool_size = pool_size

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = F.adaptive_avg_pool2d(x, self.pool_size)
        return x.flatten(1)

class QuadrantHead3Class(nn.Module):
    def __init__(self, in_features=64 * 3 * 4, num_classes=NUM_CLASSES):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

class StitchedBaselineModel(nn.Module):
    def __init__(self, crop_size=(24, 32)):
        super().__init__()
        self.backbone = ThermalBackbone(in_channels=2)
        self.head = QuadrantHead3Class()
        self.crop_size = crop_size

    def _extract_quadrant(self, x, bbox):
        x1, y1, x2, y2 = (int(round(v)) for v in bbox)
        x1, y1 = max(x1, 0), max(y1, 0)
        patch = x[:, :, y1:y2, x1:x2]
        if patch.shape[-2:] != self.crop_size:
            patch = F.interpolate(patch, size=self.crop_size, mode="nearest")
        return patch

    def forward(self, x):
        return torch.stack([self.head(self.backbone(self._extract_quadrant(x, STITCH_BBOX[c])))
                            for c in ACTIVE_CAMERAS], dim=1)

MODEL_SPECS["Stitched"] = lambda: StitchedBaselineModel()
INPUTS = {}   # model name -> (train array, val array) when it doesn't use the tile arrays
if "Stitched" in MODELS_TO_TRAIN:
    t0 = time.time()
    to_stitched = lambda rows: np.stack([build_stitched_input_tensor(load_panorama(*key_of(r))) for r in rows])
    INPUTS["Stitched"] = (to_stitched(train_rows_k), to_stitched(val_rows_k))
    print(f"stitched inputs: train {INPUTS['Stitched'][0].shape}  val {INPUTS['Stitched'][1].shape}  ({time.time() - t0:.0f}s)")
    _m = StitchedBaselineModel().eval()
    print("Stitched out", tuple(_m(torch.from_numpy(INPUTS["Stitched"][1][:4])).shape),
          f"params {sum(p.numel() for p in _m.parameters()):,}")


## 5. Training and evaluation

The loss, optimizer, schedule and best-epoch selection are the pipeline's `masked_ce_loss` and `train_classifier_model`.

In [ ]:
def masked_ce_loss(logits, labels, mask, class_weight=None):
    B, C, K = logits.shape
    per = F.cross_entropy(logits.reshape(B * C, K), labels.reshape(B * C), weight=class_weight,
                          reduction="none").reshape(B, C)
    return (per * mask).sum() / mask.sum().clamp_min(1.0)

def masked_macro_f1(preds, trues, masks, classes=(1, 2)):
    f1s = []
    for i in range(preds.shape[1]):
        v = masks[:, i] > 0
        if v.any():
            f1s.append(np.mean(f1_score(trues[v, i], preds[v, i], labels=list(classes), average=None, zero_division=0)))
    return float(np.mean(f1s)) if f1s else 0.0

def per_class_f1(preds, trues, masks):
    """{direction or 'overall': {class: F1}}. 'overall' = mean over directions, as in the paper's figures."""
    out = {}
    for i, d in enumerate(DIRECTIONS):
        v = masks[:, i] > 0
        out[d] = dict(zip(CLASS_NAMES, f1_score(trues[v, i], preds[v, i], labels=[0, 1, 2], average=None, zero_division=0)))
    out["overall"] = {c: float(np.mean([out[d][c] for d in DIRECTIONS])) for c in CLASS_NAMES}
    return out

@torch.no_grad()
def predict(model, X, perm=None):
    model.eval()
    probs = []
    for s in range(0, len(X), 512):
        x = torch.from_numpy(X[s:s + 512]).to(device)
        if perm is not None:
            x = x[:, perm]
        probs.append(F.softmax(model(x), -1).cpu())
    return torch.cat(probs).numpy()

def make_loader(X, Y, M, shuffle, seed):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M)),
                      batch_size=BATCH_SIZE, shuffle=shuffle, generator=g)   # data is in memory: no workers needed

def train(model, name, seed, Xt=None, Xv=None):
    Xt = Xtr if Xt is None else Xt
    Xv = Xva if Xv is None else Xv
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    cw = class_weight.to(device)
    tr = make_loader(Xt, Ytr, Mtr, True, seed)
    best_f1, best_w = -1.0, None
    for ep in range(EPOCHS):
        model.train()
        for x, y, m in tr:
            x, y, m = x.to(device), y.to(device), m.to(device)
            opt.zero_grad()
            masked_ce_loss(model(x), y, m, cw).backward()
            opt.step()
        sch.step()
        f1 = masked_macro_f1(predict(model, Xv).argmax(-1), Yva, Mva)
        if f1 > best_f1:
            best_f1, best_w = f1, copy.deepcopy(model.state_dict())
        if ep == 0 or (ep + 1) % 10 == 0:
            print(f"  [{name} s{seed}] ep {ep + 1:02d}/{EPOCHS}  val macro F1 {f1:.3f}")
    model.load_state_dict(best_w)
    return model

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def cpu_latency_ms(model, x_shape=(1, NUM_CAMS, 1, SENSOR_H, SENSOR_W), n=200):
    m = copy.deepcopy(model).cpu().eval()
    x = torch.zeros(*x_shape)
    for _ in range(20):
        m(x)
    t0 = time.perf_counter()
    for _ in range(n):
        m(x)
    return (time.perf_counter() - t0) / n * 1000

PERMS = [list(p) for p in itertools.permutations(range(NUM_CAMS)) if list(p) != [0, 1, 2]]
results, shuffle_results, meta = [], [], {}
(OUT / "models").mkdir(exist_ok=True)

for name, mk in MODEL_SPECS.items():
    if name not in MODELS_TO_TRAIN:
        continue
    Xt, Xv = INPUTS.get(name, (Xtr, Xva))
    for seed in TRAIN_SEEDS:
        set_seed(seed)
        t0 = time.time()
        model = train(mk(), name, seed, Xt, Xv)
        probs = predict(model, Xv)
        results.append({"model": name, "seed": seed, **{"f1": per_class_f1(probs.argmax(-1), Yva, Mva)},
                        "macro_po": masked_macro_f1(probs.argmax(-1), Yva, Mva)})
        # shuffle test: tiles AND labels permuted together; slot j is scored against the camera placed in slot j
        if name in INPUTS:   # panorama input has no camera slots to shuffle
            sh = [{c: float("nan") for c in CLASS_NAMES}]
        else:
            sh = [per_class_f1(predict(model, Xva, p).argmax(-1), Yva[:, p], Mva[:, p])["overall"] for p in PERMS]
        shuffle_results.append({"model": name, "seed": seed,
                                **{c: float(np.mean([s[c] for s in sh])) for c in CLASS_NAMES}})
        torch.save(model.state_dict(), OUT / "models" / f"{name.replace(' ', '_').replace('.', '')}_s{seed}.pt")
        if seed == TRAIN_SEEDS[0]:
            meta[name] = {"params": sum(p.numel() for p in model.parameters()),
                          "cpu_ms": cpu_latency_ms(model, (1,) + Xv.shape[1:])}
            if name == "Feature Fusion":
                FF_VAL_PROBS = probs
        print(f"{name} seed {seed}: overall {results[-1]['f1']['overall']}  ({time.time() - t0:.0f}s)")

## 6. Non-learning baselines

Each tile gets three rules:
- **person** if `max - median ≥ a` and `lo ≤ max ≤ hi` (a warm spot in the body-heat range);
- else **other** if `std ≥ b`;
- else **clear**.

The thresholds are grid-searched on **train** for **3-class** macro F1 (clear + person + other), then scored once on val.
An earlier version tuned for person + other only, so the search learned to almost never answer *clear*.

- **Threshold (non-learning)**: one set of thresholds shared by all directions.
- **Threshold (per-direction)**: each direction gets its own thresholds (12 numbers). One pass over the grid serves both.

The shuffle test works as for the learned models: slot *j* keeps its thresholds and gets another camera's tile.

In [ ]:
THRESHOLD_MODELS = ["Threshold (non-learning)", "Threshold (per-direction)"]
if any(n in MODELS_TO_TRAIN for n in THRESHOLD_MODELS):
    def tile_features(X):
        t = X[:, :, 0].reshape(len(X), NUM_CAMS, -1)
        return t.max(-1), np.median(t, -1), t.std(-1)

    def rule_predict(feats, a, lo, hi, b):   # thresholds are scalars, or one per slot (broadcast over cameras)
        tmax, tmed, tstd = feats
        person = ((tmax - tmed) >= a) & (tmax >= lo) & (tmax <= hi)
        other = ~person & (tstd >= b)
        return np.where(person, 1, np.where(other, 2, 0))

    def rule_predict_slots(feats, thr):      # thr[j] = (a, lo, hi, b) for slot j
        return rule_predict(feats, *(np.array(v) for v in zip(*thr)))

    def macro_f1_3(pred, true):              # 3-class macro F1, same as sklearn with zero_division=0, but fast
        cm = np.bincount(true * 3 + pred, minlength=9).reshape(3, 3)
        denom = cm.sum(0) + cm.sum(1)
        return float(np.mean(np.where(denom > 0, 2 * np.diag(cm) / np.maximum(denom, 1), 0.0)))

    ftr, fva = tile_features(Xtr), tile_features(Xva)
    q = lambda arr, ps: np.unique(np.percentile(arr[Mtr > 0], ps).round(2))
    combos = [c for c in itertools.product(q(ftr[0] - ftr[1], range(5, 100, 5)), q(ftr[0], range(0, 60, 10)),
                                           q(ftr[0], range(50, 101, 10)), q(ftr[2], range(5, 100, 5))) if c[1] < c[2]]
    ok = [Mtr[:, i] > 0 for i in range(NUM_CAMS)]
    f_dir = [tuple(f[ok[i], i] for f in ftr) for i in range(NUM_CAMS)]
    y_dir = [Ytr[ok[i], i] for i in range(NUM_CAMS)]
    t0 = time.time()
    scores = np.array([[macro_f1_3(rule_predict(f_dir[i], *c), y_dir[i]) for i in range(NUM_CAMS)] for c in combos])
    print(f"searched {len(combos)} threshold sets on train ({time.time() - t0:.0f}s)")

    THRESHOLDS = {"Threshold (non-learning)": [combos[scores.mean(1).argmax()]] * NUM_CAMS,
                  "Threshold (per-direction)": [combos[scores[:, i].argmax()] for i in range(NUM_CAMS)]}
    for name in THRESHOLD_MODELS:
        if name not in MODELS_TO_TRAIN:
            continue
        thr = THRESHOLDS[name]
        pred = rule_predict_slots(fva, thr)
        results.append({"model": name, "seed": 0, "f1": per_class_f1(pred, Yva, Mva),
                        "macro_po": masked_macro_f1(pred, Yva, Mva)})
        meta[name] = {"params": 4 if name == "Threshold (non-learning)" else 4 * NUM_CAMS, "cpu_ms": float("nan"),
                      "thresholds": {d: [float(v) for v in t] for d, t in zip(DIRECTIONS, thr)}}
        sh = [per_class_f1(rule_predict_slots(tuple(f[:, p] for f in fva), thr), Yva[:, p], Mva[:, p])["overall"]
              for p in PERMS]
        shuffle_results.append({"model": name, "seed": 0, **{c: float(np.mean([s[c] for s in sh])) for c in CLASS_NAMES}})
        print(f"{name}: (a, lo, hi, b) per direction {meta[name]['thresholds']}")
        print(f"  val overall F1 {({c: round(v, 3) for c, v in results[-1]['f1']['overall'].items()})}"
              f"  macro (person+other) {results[-1]['macro_po']:.3f}")

## 7. Results

In [ ]:
def mean_std(vals):
    vals = np.array(vals, dtype=float)
    if np.isnan(vals).all():
        return "—"
    return f"{vals.mean():.3f} ± {vals.std(ddof=1):.3f}" if len(vals) > 1 else f"{vals[0]:.3f}"

order = [n for n in ["Threshold (non-learning)", "Threshold (per-direction)", "Stitched", "Single-camera", "Output Fusion", "Output Fusion (FF config)", "Perm.-equivariant",
                     "Feature Fusion"] if any(r["model"] == n for r in results)]
rows = []
for name in order:
    rs = [r for r in results if r["model"] == name]
    sh = [r for r in shuffle_results if r["model"] == name]
    row = {"Model": name}
    for c in CLASS_NAMES:
        row[f"F1 {c}"] = mean_std([r["f1"]["overall"][c] for r in rs])
    row["Macro F1 (person+other)"] = mean_std([r["macro_po"] for r in rs])
    row["Macro F1 (3-class)"] = mean_std([np.mean(list(r["f1"]["overall"].values())) for r in rs])
    for c in CLASS_NAMES:
        row[f"Shuffled F1 {c}"] = mean_std([r[c] for r in sh])
    row["Params"] = f"{meta[name]['params']:,}"
    row["CPU ms"] = f"{meta[name]['cpu_ms']:.2f}"
    rows.append(row)
table = pd.DataFrame(rows)
print("Overall per-class F1 on val (mean over directions), mean ± std over seeds;")
print("'Shuffled' = averaged over the 5 non-identity camera permutations, with no correction.\n")
print(table.to_string(index=False))
table.to_csv(OUT / "baselines_table.csv", index=False)

per_dir = pd.DataFrame([{"model": r["model"], "seed": r["seed"], "direction": d, "class": c, "f1": r["f1"][d][c]}
                        for r in results for d in DIRECTIONS + ["overall"] for c in CLASS_NAMES])
per_dir.to_csv(OUT / "baselines_per_direction_per_seed.csv", index=False)
json.dump({"meta": meta, "results": results, "shuffle": shuffle_results}, open(OUT / "baselines_raw.json", "w"),
          indent=1, default=float)

print("\nPaper (cluster run), Feature Fusion overall F1 on val: clear ≈ 0.81, person ≈ 0.83, other ≈ 0.80")
print("If the retrained Feature Fusion row is far from that, the labels or split differ from the cluster run.")

## 8. Per-frame predictions for the failure-case analysis

These are Feature Fusion (seed 42) val predictions with class probabilities, one row per (frame, direction).

In [ ]:
if "Feature Fusion" in MODELS_TO_TRAIN:
    pred_rows = []
    for i, r in enumerate(val_rows_k):
        for j, d in enumerate(DIRECTIONS):
            if Mva[i, j] == 0:
                continue
            p = FF_VAL_PROBS[i, j]
            pred_rows.append({"part": r["_part"], "anchor_index": int(r["anchor_index"]), "rgb_file": r.get("rgb_files"),
                              "direction": d, "true": CLASS_NAMES[Yva[i, j]], "pred": CLASS_NAMES[p.argmax()],
                              "conf": float(p.max()), **{f"p_{c}": float(p[k]) for k, c in enumerate(CLASS_NAMES)}})
    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(OUT / "ff_val_predictions.csv", index=False)
    err = pred_df[pred_df["true"] != pred_df["pred"]]
    print(f"{len(pred_df)} labeled val cells, {len(err)} errors ({len(err) / len(pred_df):.1%})")
    print(pd.crosstab(err["true"], err["pred"]).to_string())